# 3. Couches raster (pixel par pixel)

GeoTIFF publiés dans `data/api/layers/<indicateur>/` (grille EPSG:4326) :

- `AAAA.tif` : valeur annuelle (cumul pour la pluie, moyenne sinon), seulement pour les années complètes ;
- `baseline.tif` : moyenne de référence 2001-2020 ;
- `anomaly_AAAA.tif` : écart à la référence pour une année ;
- `elevation/elevation.tif` : altitude NASADEM (affichée sous-échantillonnée).

Les contours (pays, régions ou préfectures) et les sites pilotes sont superposés.

In [ ]:
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as w
from IPython.display import JSON, Markdown, display

import agbea_nb as nb

pd.set_option("display.max_rows", 60)
cat = nb.catalog()
print(f"Données : {nb.API}")
print(f"Générées le {cat['generated_at']} · pipeline {cat['pipeline_version']} · référence {cat['baseline']}")

import rasterio

LAYERS = nb.API / "layers"
print("Couches :", {d.name: len(list(d.glob('*.tif'))) for d in sorted(LAYERS.iterdir()) if d.is_dir()})

In [ ]:
def fichiers(ind):
    names = sorted(p.name for p in (LAYERS / ind).glob("*.tif"))
    order = lambda n: (0 if n == "baseline.tif" else 1 if n[0].isdigit() else 2, n)
    return sorted(names, key=order)


def lire(path, max_rows=900):
    """Lit la couche (sous-échantillonnée si besoin pour l'affichage) et ses coordonnées de centres de pixels."""
    with rasterio.open(path) as src:
        step = max(1, int(np.ceil(src.height / max_rows)))
        arr = src.read(1, masked=True)[::step, ::step].astype("float32").filled(np.nan)
        t = src.transform
        lon = t.c + (np.arange(0, src.width, step) + 0.5) * t.a
        lat = t.f + (np.arange(0, src.height, step) + 0.5) * t.e
        meta = {"taille": f"{src.width} × {src.height} px", "pas affiché": step, "résolution (°)": round(t.a, 5),
                "crs": str(src.crs), **src.tags()}
    return arr, lon, lat, meta


def couche(ind, fichier, contours, sites):
    arr, lon, lat, meta = lire(LAYERS / ind / fichier)
    unit = cat["indicators"].get(ind, {}).get("unit", "")
    seq, div = nb.SCALES.get(ind, ("Viridis", "RdBu_r"))
    anomaly = fichier.startswith("anomaly")
    fig = go.Figure(go.Heatmap(z=arr, x=lon, y=lat, colorscale=div if anomaly else seq, zmid=0 if anomaly else None,
                               colorbar=dict(title=unit), hovertemplate="lon %{x:.3f}, lat %{y:.3f}<br>%{z:.3g} " + unit + "<extra></extra>"))
    if contours:
        xs, ys = nb.boundary_lines(contours)
        fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color="black", width=0.8), hoverinfo="skip"))
    if sites:
        s = nb.units().query("level == 'site'")
        fig.add_trace(go.Scatter(x=s.centroid_lon, y=s.centroid_lat, mode="markers+text", text=s.name,
                                 textposition="top right", marker=dict(color="black", size=7), hoverinfo="text"))
    fig.update_yaxes(scaleanchor="x", scaleratio=1, title="latitude")
    fig.update_xaxes(title="longitude", range=[float(lon.min()), float(lon.max())])
    titre = {"baseline.tif": f"moyenne de référence {cat['baseline']}"}.get(fichier, fichier.removesuffix(".tif").replace("anomaly_", "écart à la référence, "))
    fig.update_layout(title=f"{nb.label(ind)} — {titre}", height=900, width=620, showlegend=False,
                      margin=dict(l=60, r=20, t=60, b=50))
    fig.show()
    v = arr[np.isfinite(arr)]
    stats = {"pixels valides": f"{v.size / arr.size:.0%}", "min": float(v.min()), "médiane": float(np.median(v)),
             "moyenne": float(v.mean()), "max": float(v.max())} if v.size else {"pixels valides": "0 %"}
    display(pd.Series({**stats, **meta}, name=f"{ind}/{fichier}").to_frame())

## Choisir une couche

In [ ]:
choix_ind = w.Dropdown(options=[(nb.label(d.name), d.name) for d in sorted(LAYERS.iterdir()) if d.is_dir()],
                       value="rainfall", description="Indicateur")
choix_fichier = w.Dropdown(options=fichiers("rainfall"), description="Couche")
choix_contours = w.Dropdown(options=[("Régions", "region"), ("Préfectures", "prefecture"), ("Pays", "country"), ("Aucun", None)],
                            value="region", description="Contours")
choix_sites = w.Checkbox(value=True, description="Sites pilotes")


def _maj_fichiers(change):
    choix_fichier.options = fichiers(change["new"])

choix_ind.observe(_maj_fichiers, names="value")
sortie = w.interactive_output(couche, {"ind": choix_ind, "fichier": choix_fichier, "contours": choix_contours, "sites": choix_sites})
display(w.HBox([choix_ind, choix_fichier, choix_contours, choix_sites]), sortie)